# Sequence models for text

Every language model learns one thing: given what came before, what comes next? This notebook builds that objective from scratch as a character-level prediction task. The model reads a sequence of characters and predicts the next one at each position. We start with a vanilla RNN coded cell by cell, then swap in an LSTM and compare the two.

In [ ]:
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from nltk.corpus import gutenberg
from torch import nn

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"device: {device}")

## 1. Intuition

Autocomplete on a phone is a language model. It has seen billions of phrases and learned that "Happy birth" is almost always followed by "day". A character-level model does the same thing one character at a time — "Happ" → "y", "Happy" → " ", and so on. The task is well-defined: at every position in the training text, the correct answer is the next character.

An RNN handles this by keeping a hidden state: a fixed-size vector that summarises everything seen so far. At each step it updates the hidden state using the new input and the previous state, then predicts the next character from the updated state. The same weights are reused at every step, which is what makes the model a *recurrent* network.

## 2. Maths

The RNN recurrence:

$$h_t = \tanh(W_{xh}\, x_t + W_{hh}\, h_{t-1} + b)$$

where $x_t$ is the input embedding at step $t$ and $h_t$ is the hidden state. A linear output layer then maps $h_t$ to logits over the vocabulary.

The training objective is cross-entropy summed over all positions:

$$\mathcal{L} = -\frac{1}{T}\sum_{t=1}^{T} \log P(c_t \mid c_{1:t-1})$$

**Perplexity** is $\exp(\mathcal{L})$ — the effective vocabulary size the model is uncertain about at each step. A perplexity of 5 means the model is as uncertain as if it were choosing uniformly from 5 characters.

## 3. Data

Blake's poems from NLTK Gutenberg — 38 K characters, already downloaded. We use the first 8 K to keep training fast. The vocabulary is every unique character in that slice.

In [ ]:
raw = gutenberg.raw("blake-poems.txt")[:8000]
chars = sorted(set(raw))
vocab_size = len(chars)
char_to_ix = {c: i for i, c in enumerate(chars)}
ix_to_char = {i: c for i, c in enumerate(chars)}

data = torch.tensor([char_to_ix[c] for c in raw], dtype=torch.long)
print(f"text length: {len(raw)}, vocab size: {vocab_size}")
print(f"sample: {raw[:60]!r}")

In [ ]:
def get_batch(
    data: torch.Tensor, seq_len: int, batch_size: int, rng: np.random.Generator
) -> tuple[torch.Tensor, torch.Tensor]:
    """Sample batch_size random (input, target) pairs; target is input shifted by 1."""
    n = len(data)
    starts = rng.integers(0, n - seq_len - 1, size=batch_size)
    X = torch.stack([data[s : s + seq_len] for s in starts])
    Y = torch.stack([data[s + 1 : s + seq_len + 1] for s in starts])
    return X.to(device), Y.to(device)


_rng = np.random.default_rng(SEED)
Xb, Yb = get_batch(data, seq_len=40, batch_size=4, rng=_rng)
print(f"X: {Xb.shape}, Y: {Yb.shape}")
print(f"input:  {''.join(ix_to_char[i.item()] for i in Xb[0][:30])!r}")
print(f"target: {''.join(ix_to_char[i.item()] for i in Yb[0][:30])!r}")

## 4. Vanilla RNN from scratch

The two weight matrices ($W_{xh}$ and $W_{hh}$) are plain `nn.Linear` layers. The forward pass steps through the sequence one character at a time, updating the hidden state at each step. This makes the recurrence visible in the code.

In [ ]:
class CharRNN(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int, hidden_size: int) -> None:
        super().__init__()
        self.hidden_size = hidden_size
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.W_xh = nn.Linear(embed_dim, hidden_size, bias=False)
        self.W_hh = nn.Linear(hidden_size, hidden_size, bias=True)
        self.out = nn.Linear(hidden_size, vocab_size)

    def forward(
        self, x: torch.Tensor, h: torch.Tensor
    ) -> tuple[torch.Tensor, torch.Tensor]:
        # x: (batch, seq_len), h: (batch, hidden_size)
        emb = self.embed(x)  # (batch, seq_len, embed_dim)
        logits = []
        for t in range(emb.size(1)):
            h = torch.tanh(self.W_xh(emb[:, t]) + self.W_hh(h))
            logits.append(self.out(h))
        return torch.stack(logits, dim=1), h  # (batch, seq_len, vocab_size), (batch, hidden)

    def init_hidden(self, batch_size: int) -> torch.Tensor:
        return torch.zeros(batch_size, self.hidden_size, device=device)

Training: Adam with gradient clipping. Clipping prevents the hidden state from exploding when gradients flow back through many time steps.

In [ ]:
SEQ_LEN = 40
BATCH_SIZE = 32
EMBED_DIM = 32
HIDDEN_SIZE = 128
EPOCHS = 20
STEPS = 10  # gradient steps per epoch
LR = 1e-3

torch.manual_seed(SEED)
rnn = CharRNN(vocab_size, EMBED_DIM, HIDDEN_SIZE).to(device)
rnn_opt = torch.optim.Adam(rnn.parameters(), lr=LR)
rnn_rng = np.random.default_rng(SEED)

rnn_losses: list[float] = []
for epoch in range(1, EPOCHS + 1):
    rnn.train()
    epoch_loss = 0.0
    for _ in range(STEPS):
        X, Y = get_batch(data, SEQ_LEN, BATCH_SIZE, rnn_rng)
        h = rnn.init_hidden(BATCH_SIZE)
        logits, _ = rnn(X, h)
        loss = F.cross_entropy(logits.reshape(-1, vocab_size), Y.reshape(-1))
        rnn_opt.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(rnn.parameters(), 1.0)
        rnn_opt.step()
        epoch_loss += loss.item()
    avg = epoch_loss / STEPS
    rnn_losses.append(avg)
    if epoch % 5 == 0:
        print(f"epoch {epoch:3d}  loss {avg:.4f}  perplexity {np.exp(avg):.1f}")

In [ ]:
def sample_rnn(
    model: CharRNN, seed_text: str, n_chars: int = 150, temperature: float = 0.8
) -> str:
    """Autoregressively sample characters from a CharRNN."""
    model.eval()
    out = list(seed_text)
    h = model.init_hidden(1)
    with torch.no_grad():
        for c in seed_text[:-1]:  # warm up hidden state
            x = torch.tensor([[char_to_ix.get(c, 0)]], device=device)
            _, h = model(x, h)
        for _ in range(n_chars):
            x = torch.tensor([[char_to_ix.get(out[-1], 0)]], device=device)
            logits, h = model(x, h)
            probs = F.softmax(logits[0, 0] / temperature, dim=0)
            ix = torch.multinomial(probs, 1).item()
            out.append(ix_to_char[ix])
    return "".join(out)


torch.manual_seed(SEED)
print("RNN sample:")
print(sample_rnn(rnn, "The ", temperature=0.8))

## 5. LSTM version

`nn.LSTM` replaces the two manual weight matrices with four gated transformations per step (forget, input, cell, output). The cell state $c_t$ gives the LSTM a separate long-range memory track, which helps when relevant context is many steps back. The training loop is identical.

In [ ]:
class CharLSTM(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int, hidden_size: int) -> None:
        super().__init__()
        self.hidden_size = hidden_size
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_size, batch_first=True)
        self.out = nn.Linear(hidden_size, vocab_size)

    def forward(
        self, x: torch.Tensor, state: tuple[torch.Tensor, torch.Tensor]
    ) -> tuple[torch.Tensor, tuple[torch.Tensor, torch.Tensor]]:
        h, state = self.lstm(self.embed(x), state)
        return self.out(h), state

    def init_hidden(
        self, batch_size: int
    ) -> tuple[torch.Tensor, torch.Tensor]:
        z = torch.zeros(1, batch_size, self.hidden_size, device=device)
        return z, z.clone()

In [ ]:
torch.manual_seed(SEED)
lstm = CharLSTM(vocab_size, EMBED_DIM, HIDDEN_SIZE).to(device)
lstm_opt = torch.optim.Adam(lstm.parameters(), lr=LR)
lstm_rng = np.random.default_rng(SEED)

lstm_losses: list[float] = []
for epoch in range(1, EPOCHS + 1):
    lstm.train()
    epoch_loss = 0.0
    for _ in range(STEPS):
        X, Y = get_batch(data, SEQ_LEN, BATCH_SIZE, lstm_rng)
        state = lstm.init_hidden(BATCH_SIZE)
        logits, _ = lstm(X, state)
        loss = F.cross_entropy(logits.reshape(-1, vocab_size), Y.reshape(-1))
        lstm_opt.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(lstm.parameters(), 1.0)
        lstm_opt.step()
        epoch_loss += loss.item()
    avg = epoch_loss / STEPS
    lstm_losses.append(avg)
    if epoch % 5 == 0:
        print(f"epoch {epoch:3d}  loss {avg:.4f}  perplexity {np.exp(avg):.1f}")

print(f"\nFinal — RNN: {np.exp(rnn_losses[-1]):.1f}  LSTM: {np.exp(lstm_losses[-1]):.1f}")

In [ ]:
def sample_lstm(
    model: CharLSTM, seed_text: str, n_chars: int = 150, temperature: float = 0.8
) -> str:
    model.eval()
    out = list(seed_text)
    state = model.init_hidden(1)
    with torch.no_grad():
        for c in seed_text[:-1]:
            x = torch.tensor([[char_to_ix.get(c, 0)]], device=device)
            _, state = model(x, state)
        for _ in range(n_chars):
            x = torch.tensor([[char_to_ix.get(out[-1], 0)]], device=device)
            logits, state = model(x, state)
            probs = F.softmax(logits[0, 0] / temperature, dim=0)
            ix = torch.multinomial(probs, 1).item()
            out.append(ix_to_char[ix])
    return "".join(out)


torch.manual_seed(SEED)
print("LSTM sample:")
print(sample_lstm(lstm, "The ", temperature=0.8))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(rnn_losses, label="RNN")
ax.plot(lstm_losses, label="LSTM")
ax.set_xlabel("epoch")
ax.set_ylabel("cross-entropy loss")
ax.set_title("Training loss: RNN vs LSTM")
ax.legend()
plt.tight_layout()
plt.show()

## 6. Tuning

- **`seq_len`** controls how far back the model can see within one training window. Longer sequences give more context but make training slower and gradients harder to propagate.
- **`hidden_size`** sets the capacity. Doubling it from 128 to 256 usually lowers perplexity but roughly doubles training time.
- **Temperature** at sampling time scales the logits before softmax. Values below 1.0 make the distribution sharper (more repetitive, safer output). Values above 1.0 increase diversity and noise.

## 7. What breaks

- **Vanishing gradients in vanilla RNN.** The gradient of the loss with respect to $h_0$ involves a product of Jacobians, one per time step. Each is bounded by the tanh derivative (at most 1), so the signal disappears over long sequences. The LSTM's cell state provides an additive gradient path that sidesteps this.
- **Exploding gradients.** The same product can grow if the spectral radius of $W_{hh}$ exceeds 1. Gradient clipping (`clip_grad_norm_`) caps the global norm of the gradient vector before each update.
- **Memorisation on small corpora.** With only 8 K characters the model can overfit and reproduce training text verbatim. Good language modelling requires far more data than this. The numbers here show mechanics, not production quality.